# 06 - Query `FAFBMCNetwork` Demo

This notebook is a lightweight demo for querying neuron-level information from the optic-lobe `FAFBMCNetwork`.

It focuses on:
- global network summary
- finding neurons by `root_id` or `cell_type`
- reading HH parameters for one neuron
- checking morphology-related metadata
- inspecting incoming and outgoing synapses
- building small tables for quick debugging


In [ ]:
import numpy as np
import pandas as pd

import sys, os
sys.path.insert(0, os.path.abspath('../..'))

from neuro_framework.models import (
    load_or_build_cached_net,
    apply_postbuild_parameter_overrides,
    build_pathway_override_rules,
    net_summary,
    neuron_index_from_root_id,
    neuron_indices_from_type,
    neuron_row,
    edge_table_for_neuron,
)

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)


## Paths and build config

This notebook rebuilds the optic-lobe network so it can be run independently.
If you already have a `net` object in another notebook kernel, you can skip the build cell and reuse that object instead.


In [ ]:
DATA_DIR = '../../mcHH/data/optic_lobe_right'
MORPH_PKG_DIR = '../../mcHH/data/optic_lobe_type_packages_v1'
ION_RULES = '../data/ion_channel_rules.csv'
SYN_RULES = '../data/synapse_rules.csv'
NT_ION_RULES = None
ROOT_ION_OVERRIDES = None

DT = 0.1
NCOMP = 2
MIN_SYN_COUNT = 3
MORPH_PROGRESS_EVERY = 5000
R16_ELEAK_SHIFT_MV = -10.0
R16_VTH_SHIFT_MV = 8.0
R16_TARGET_VTH_SHIFTS_MV = {'L1': -4.0, 'L3': -4.0}
R16_TARGET_GS_GAINS = {'L1': 1.5, 'L3': 1.25}

CACHE_DIR = '../../mcHH/data/cache'
NET_CACHE_PATH = f'{CACHE_DIR}/optic_lobe_net_type_rules_v3.pt'
FORCE_REBUILD = False
SAVE_CACHE_AFTER_BUILD = True

print('DATA_DIR           =', DATA_DIR)
print('MORPH_PKG_DIR      =', MORPH_PKG_DIR)
print('ION_RULES          =', ION_RULES)
print('SYN_RULES          =', SYN_RULES)
print('NET_CACHE_PATH     =', NET_CACHE_PATH)


## Build the network

This can take a few minutes because the optic-lobe build includes per-neuron morphology packaging and the large sparse morphology solver.


In [ ]:
cache_meta = {
    'data_dir': DATA_DIR,
    'morph_pkg_dir': MORPH_PKG_DIR,
    'ion_rules': ION_RULES,
    'syn_rules': SYN_RULES,
    'dt': DT,
    'ncomp': NCOMP,
    'min_syn_count': MIN_SYN_COUNT,
}

net, loaded_cache_meta, loaded_from_cache, build_elapsed = load_or_build_cached_net(
    cache_path=NET_CACHE_PATH,
    force_rebuild=FORCE_REBUILD,
    save_cache_after_build=SAVE_CACHE_AFTER_BUILD,
    cache_meta=cache_meta,
    data_dir=DATA_DIR,
    morphology_package_dir=MORPH_PKG_DIR,
    ion_rules_path=ION_RULES,
    syn_rules_path=SYN_RULES,
    nt_ion_rules_path=NT_ION_RULES,
    neuron_ion_overrides_path=ROOT_ION_OVERRIDES,
    dt=DT,
    ncomp=NCOMP,
    min_syn_count=MIN_SYN_COUNT,
    morphology_progress_every=MORPH_PROGRESS_EVERY,
)
neuron_override_rules, synapse_override_rules = build_pathway_override_rules(
    eLeak_shift_mV=R16_ELEAK_SHIFT_MV,
    v_th_shift_mV=R16_VTH_SHIFT_MV,
    target_v_th_shifts_mV=R16_TARGET_VTH_SHIFTS_MV,
    target_gs_gains=R16_TARGET_GS_GAINS,
)
net = apply_postbuild_parameter_overrides(
    net, neuron_rules=neuron_override_rules, synapse_rules=synapse_override_rules, reset_first=True
)
print('loaded_from_cache =', loaded_from_cache)
print(f'build_elapsed = {build_elapsed:.1f}s')
print(net)
print('cache_meta =', loaded_cache_meta)


## 1. Global summary

This is the fastest way to understand what is inside the current `net` object.


In [ ]:
summary = net_summary(net)
summary


In [ ]:
type_counts = pd.Series(net.cell_types).value_counts().rename_axis('cell_type').reset_index(name='count')
display(type_counts.head(20))


## 2. Helper functions

The helpers below make it easy to look up one neuron and inspect its parameters, morphology metadata, and connectivity.


In [ ]:
# Query helpers are imported from neuro_framework.models.fafb_notebook_helpers
print('Imported helpers: neuron_index_from_root_id, neuron_indices_from_type, neuron_row, edge_table_for_neuron')


## 3. Query by `root_id`

Replace the example `root_id` below with any neuron you care about.


In [ ]:
EXAMPLE_ROOT_ID = int(net.root_ids[0])
example_idx = neuron_index_from_root_id(net, EXAMPLE_ROOT_ID)
print('example_idx =', example_idx)
print('example_root_id =', EXAMPLE_ROOT_ID)

display(neuron_row(net, example_idx).to_frame('value'))


## 4. Query by `cell_type`

This is useful when you want a few example neurons from one type.


In [ ]:
EXAMPLE_TYPE = 'R1-6'
example_type_indices = neuron_indices_from_type(net, EXAMPLE_TYPE, limit=10)
print('example_type_indices =', example_type_indices[:10])

rows = pd.DataFrame([neuron_row(net, i) for i in example_type_indices[:5]])
display(rows)


## 5. Inspect one neuron's connectivity

This shows the first few outgoing and incoming edges for one neuron.


In [ ]:
target_idx = example_type_indices[0] if example_type_indices else example_idx
print('target neuron:', target_idx, net.cell_types[target_idx], int(net.root_ids[target_idx]))

out_edges = edge_table_for_neuron(net, target_idx, direction='out', limit=15)
in_edges = edge_table_for_neuron(net, target_idx, direction='in', limit=15)

print('Outgoing edges')
display(out_edges)
print('Incoming edges')
display(in_edges)


## 6. Type-level summary of HH parameters

This is useful for checking whether a neuron type is using the expected initialization.


In [ ]:
per_neuron_df = pd.DataFrame({
    'root_id': net.root_ids,
    'cell_type': net.cell_types,
    'resolved_cell_type': getattr(net, 'resolved_cell_types', net.cell_types),
    'dominant_nt': getattr(net, 'neuron_nt_types', [None] * net.n_neurons),
    'param_source': getattr(net, 'neuron_param_source', [None] * net.n_neurons),
    'fallback_target': getattr(net, 'neuron_fallback_target', [None] * net.n_neurons),
    'gNa_mS_cm2': net.g_Na.detach().cpu().numpy(),
    'gK_mS_cm2': net.g_K.detach().cpu().numpy(),
    'gLeak_mS_cm2': net.g_L.detach().cpu().numpy(),
    'eNa_mV': net.E_Na.detach().cpu().numpy(),
    'eK_mV': net.E_K.detach().cpu().numpy(),
    'eLeak_mV': net.E_L.detach().cpu().numpy(),
    'n_solver_compartments': net.neuron_n_comp.detach().cpu().numpy() if hasattr(net, 'neuron_n_comp') else np.full(net.n_neurons, net.n_comp),
    'n_swc_nodes': getattr(net, 'morphology_node_counts', np.full(net.n_neurons, np.nan)),
})

type_param_summary = per_neuron_df.groupby('cell_type', dropna=False).agg(
    n_neurons=('root_id', 'size'),
    dominant_nt_mode=('dominant_nt', lambda s: s.mode().iloc[0] if not s.mode().empty else None),
    param_source_mode=('param_source', lambda s: s.mode().iloc[0] if not s.mode().empty else None),
    gNa_mean=('gNa_mS_cm2', 'mean'),
    gK_mean=('gK_mS_cm2', 'mean'),
    gLeak_mean=('gLeak_mS_cm2', 'mean'),
    eLeak_mean=('eLeak_mV', 'mean'),
    n_comp_mean=('n_solver_compartments', 'mean'),
    n_nodes_mean=('n_swc_nodes', 'mean'),
).reset_index()

display(type_param_summary.sort_values(['n_neurons', 'cell_type'], ascending=[False, True]).head(20))


## 7. Quick recipes

Below are a few short patterns you can reuse in other notebooks.


In [ ]:
# Query one neuron by root_id
idx = neuron_index_from_root_id(net, int(net.root_ids[100]))
print(neuron_row(net, idx))

# Query all neuron indices for one type (show first 10)
print(neuron_indices_from_type(net, 'Mi1', limit=10))

# Check which parameter source a type is using
print(per_neuron_df.loc[per_neuron_df['cell_type'] == 'R1-6', ['root_id', 'param_source', 'dominant_nt', 'eLeak_mV']].head())

# Check outgoing synapses for one neuron
example_out = edge_table_for_neuron(net, idx, direction='out', limit=5)
display(example_out)


## 8. Optional export

If you want a persistent table for later analysis, export `per_neuron_df` or any filtered subset.


In [ ]:
# Example export
# per_neuron_df.to_csv('optic_lobe_per_neuron_query_demo.csv', index=False)
# type_param_summary.to_csv('optic_lobe_type_query_demo.csv', index=False)
